In [3]:
import yaml
import json

In [4]:
class helperfunction():
    def load_file(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return f.read()
    def load_yaml(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return yaml.safe_load(f)
    def fop(self,float_num):
        return float(f"{float_num:.1f}")
    def jsonstr(self,ip):
        return str(json.dumps(ip,indent=4, ensure_ascii=False))

In [5]:
hp = helperfunction()
resume_json = hp.load_file("resume_json.txt")
print(resume_json)

{
  "contact_information": {
    "name": "Surya Teja Menta",
    "email": "-",
    "phone": "+91 8309584461",
    "linkedin": "-",
    "jobdb_link": "-",
    "portfolio_link": "suryatejamenta.co.in"
  },
  "professional_summary": {
    "has_summary": "Yes",
    "summary_points": [
      "I’m Surya Teja Menta, Results-driven Senior Data Scientist with 4+ years of experience in Data Science, Machine Learning (ML), and Generative AI (GenAI).",
      "Proven expertise in RAG (Retrieval-Augmented Generation), LLM fine-tuning, MLOps, and end-to-end AI solutions.",
      "Strong background in data analytics, statistical modeling, AI-powered automation, and scalable AI architectures.",
      "IBM Certified Professional Data Scientist with hands-on experience in LangChain, Hugging Face, OpenAI APIs, Vector Databases (ChromaDB, Pinecone), and cloud deployments (AWS, GCP).",
      "Passionate about AI research, model optimization, and developing cutting-edge AI solutions."
    ]
  },
  "education

In [6]:
from google import genai

In [11]:
import json

class PromptBuilder(helperfunction):
    def __init__(self, section, criteria, targetrole, cvresume, include_fewshot: bool = True):
        self.section        = section
        self.criteria       = criteria[::-1]
        self.cvresume       = cvresume
        self.targetrole     = targetrole
        self.include_fewshot = include_fewshot
        
        self.config = self.load_yaml("prompt2.yaml")
        self.criteria_cfg = self.config.get("criteria", {})

    def build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            }
        }
    
    def _build_criteria_block(self) -> str:
        blocks = []
        for crit in self.criteria:
            block = f"- {crit}\n"
            if self.include_fewshot and crit in self.criteria_cfg:
                few_cfg = self.criteria_cfg[crit]
                for score in [5, 3, 1]:
                    key = f"score{score}"
                    if key in few_cfg:
                        text = few_cfg[key].strip()
                        block += f"    score {score}: {text}\n"
            blocks.append(block)
        return "".join(blocks)
    
    def build(self):
        config_role      = self.config['role']['role1']
        config_objective = self.config['objective']['objective1']
        config_section   = self.config['section']['section1']
        config_expected  = self.config['expected_content'][self.section]
        config_scale     = self.config['scale']['score1']
        criteria_block = self._build_criteria_block()
        prompt_role      = f"Role :\n{config_role}\n\n"
        prompt_objective = f"objectvie :\n{config_objective}\n"
        prompt_section   = f"section :\n{config_section}\n\n"
        prompt_expected  = f"expected :\n{config_expected}\n"
        prompt_criteria  = f"Criteria :\n{criteria_block}\n"
        prompt_scale     = f"Scale :\n{config_scale}\n"
        prompt_output    = f"output :\n{json.dumps(self.build_response_template(), indent=2)}\n\n"
        prompt_cvresume  = f"CV/Resume: \n{self.cvresume}\n"
        prompt = (
            prompt_role + prompt_objective + prompt_section
            + prompt_expected + prompt_criteria + prompt_scale
            + prompt_output + prompt_cvresume
        )
        prompt = prompt.replace("<section_name>", self.section)
        prompt = prompt.replace("<targetrole>", self.targetrole)
        return prompt


In [12]:
p1 = PromptBuilder(
    section="Experience",
    criteria=["RoleRelevance", "ContentQuality", "Completeness"],
    targetrole="Senior Data Scientist",
    cvresume = resume_json
)
prompt1 = p1.build()
print(prompt1)


Role :
You are the expert HR evaluator

objectvie :
Evaluate the Experience section from the resume using the scoring criteria
Measure how well the candidate matches the Senior Data Scientist role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Experience section.

expected :
- Job title, employer, dates
- Clear bullet points
- Action → method → impact structure
- Technical tools used
- Quantifiable metrics
- Feedback word 20 words

Criteria :
- Completeness
    score 5: Section contains all key elements from expected_content for this section with enoughdetail to understand the candidate's background and context. No major information gaps.
    score 3: Section contains all key elements from expected_content for this section with enoughdetail to understand the candidate's background and context. No major information gaps.
    score 1: Section is very sparse or missing most key ele

In [33]:
p2 = PromptBuilder( 
    section  = "Summary", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt2 = p2.build()
print(prompt2)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Summary section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Summary section.

expected :
- 2-4 sentence summary of experience
- Technical & domain strengths
- Career focus & value proposition
- Avoid buzzwords
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilitiesclearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities,but mixed with less relevant tasks or not yet at the depth/seniority typically expected forData science.
    score 1: Content is mostly unrelated t

In [34]:
p3 = PromptBuilder( 
    section  = "Education", 
    criteria = ["Completeness","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt3 = p3.build()
print(prompt3)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Education section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Education section.

expected :
- Institution name
- Degree & field of study
- Dates attended
- GPA, honors (optional)
- Relevance to data career
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilitiesclearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities,but mixed with less relevant tasks or not yet at the depth/seniority typically expected forData science.
    score 1: Content is mostly unrelated to t

In [35]:
p4 = PromptBuilder( 
    section    = "Experience", 
    criteria   = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt4 = p4.build()
print(prompt4)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Experience section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Experience section.

expected :
- Job title, employer, dates
- Clear bullet points
- Action → method → impact structure
- Technical tools used
- Quantifiable metrics
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilitiesclearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities,but mixed with less relevant tasks or not yet at the depth/seniority typically expected forData science.
    score 1: Content is

In [36]:
p5 = PromptBuilder( 
    section  = "Activities", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt5 = p5.build()
print(prompt5)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Activities section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Activities section.

expected :
- Competitions, hackathons, club activities
- Project descriptions with responsibilities
- Mention of tools/tech if applicable
- Feedback word 20 words

Criteria :
- Length
    score 5: Length is appropriate for the section: not too short, not overly long; information is denseand relevant; each sentence or bullet adds value without obvious redundancy.
    score 3: Length is somewhat suboptimal: either a bit short (missing some detail) or somewhat long withmild repetition or low-value bullets, but still usable.
    score 1: Length is clearly inappropriate: either extremely short (1-2 vague lines) or very long andrepet

In [37]:
p6 = PromptBuilder( 
    section  = "Skills", 
    criteria = ["Completeness","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt6 = p6.build()
print(prompt6)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Skills section from the resume using the scoring criteria
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale above.

section :
You are evaluating the Skills section.

expected :
- Technical skills (Python, SQL, ML, Cloud)
- Tools (Power BI, Git, TensorFlow)
- Soft skills
- Language proficiency
- Clear grouping/categorization
- Feedback word 20 words

Criteria :
- RoleRelevance
    score 5: Content is strongly aligned with the target role: tasks, tools, domain, and responsibilitiesclearly match what is expected for the Data science role; shows appropriate seniority and impact.
    score 3: Content has partial alignment with the target role: some relevant skills or responsibilities,but mixed with less relevant tasks or not yet at the depth/seniority typically expected forData science.
    score 1: Con

In [38]:
import re
import json

In [39]:
class LlmCaller(helperfunction):
    def __init__(self):
        GOOGLE_API_KEY = self.load_yaml("global.yaml")["setting"]["GOOGLE_API_KEY"]
        self.client    = genai.Client(api_key=GOOGLE_API_KEY)
        self.model     = self.load_yaml("model.yaml")["model"]["generation_model"]
    def parser(self,resp):
        text = resp.text.strip()
        text = re.sub(r"^```json|```$", "", text).strip()
        data = json.loads(text)
        return data
    def hit(self,prompt):
        self.prompt = prompt
        self.resp = self.client.models.generate_content(
            model    = self.model,
            contents = self.prompt
            )
        data = self.parser(self.resp)
        return data

In [40]:
op = LlmCaller()

In [41]:
op1 = op.hit(prompt1)
op2 = op.hit(prompt2)
op3 = op.hit(prompt3)
op4 = op.hit(prompt4)
op5 = op.hit(prompt5)
op6 = op.hit(prompt6)

In [43]:
op1

{'section': 'Experience',
 'scores': {'Completeness': {'score': 5,
   'feedback': 'The section clearly presents job titles, employers, dates for each role, and uses well-structured bullet points to describe responsibilities and projects, leaving no major information gaps.'},
  'ContentQuality': {'score': 3,
   'feedback': 'While the bullet points effectively describe actions and specific technical methods/tools used (e.g., YOLOv8, RAG pipelines, NLP Transformers, PyTorch, TensorFlow), there is a consistent lack of quantifiable impact or results for the technical achievements. Adding specific metrics (e.g., model accuracy improvements, efficiency gains, percentage reduction in errors, stakeholder feedback) would significantly strengthen the content quality by demonstrating measurable outcomes.'},
  'RoleRelevance': {'score': 5,
   'feedback': 'The experience section is highly relevant to a Senior Data Scientist role. It demonstrates strong alignment in advanced tasks (e.g., Generative A

In [44]:
op2

{'section': 'Summary',
 'scores': {'RoleRelevance': {'score': 5,
   'feedback': 'Clearly aligns with a Senior Data Scientist role, highlighting relevant experience, tools, and advanced AI concepts.'},
  'Length': {'score': 3,
   'feedback': 'Slightly exceeds the 2-4 sentence guideline, but each point adds value without being overtly repetitive.'},
  'Grammar': {'score': 5,
   'feedback': 'Excellent grammar, spelling, and sentence structure; the summary is well-written and easy to understand.'},
  'ContentQuality': {'score': 3,
   'feedback': 'Provides strong technical specifics and tools but could be enhanced by including quantifiable impacts or achievements.'},
  'Completeness': {'score': 5,
   'feedback': 'Contains all expected elements, including experience, technical strengths, and career focus, in good detail.'}}}

In [45]:
op3

{'section': 'Education',
 'scores': {'RoleRelevance': {'score': 4,
   'feedback': "Bachelor's in Computer Science is a strong foundational degree, highly relevant for a Data Science career path."},
  'Completeness': {'score': 5,
   'feedback': 'All key education elements, including institution, degree, dates, and GPA, are comprehensively listed.'}}}

In [46]:
op4

{'section': 'Experience',
 'scores': {'RoleRelevance': {'score': 5,
   'feedback': "The experience section strongly aligns with the target Data Science role. The candidate's progression from 'Subject Matter Expert (Data Analyst)' to 'Senior Data Scientist' shows clear career growth. The descriptions include highly relevant data science tasks, tools (YOLOv8, RAG, LLM fine-tuning, MLOps, NLP Transformers), and responsibilities that match the expectations for a senior data scientist, evidencing appropriate seniority and impact."},
  'Length': {'score': 5,
   'feedback': "The length is appropriate and well-balanced. Each role has 5-6 concise bullet points that are dense with relevant information. There is no redundancy, and the section provides a comprehensive overview of the candidate's professional background without being overly long or too short."},
  'Grammar': {'score': 5,
   'feedback': 'The grammar, spelling, and sentence structure throughout the experience section are excellent. A

In [47]:
op5

{'section': 'Activities',
 'scores': {'Length': {'score': 0,
   'feedback': "The 'Activities' section is entirely missing from the provided resume."},
  'Grammar': {'score': 0,
   'feedback': "The 'Activities' section is entirely missing from the provided resume, so grammar cannot be evaluated."},
  'ContentQuality': {'score': 0,
   'feedback': "The 'Activities' section is entirely missing from the provided resume, so content quality cannot be evaluated."},
  'Completeness': {'score': 0,
   'feedback': "The 'Activities' section is entirely missing from the provided resume. It lacks all expected elements such as competitions, hackathons, club activities, and project descriptions."}}}

In [49]:
op6

{'section': 'Skills',
 'scores': {'RoleRelevance': {'score': 5,
   'feedback': 'Excellent alignment, covering advanced ML/GenAI, MLOps, and foundational skills, demonstrating strong seniority.'},
  'Length': {'score': 5,
   'feedback': 'Comprehensive and well-detailed, covering a wide range of relevant skills efficiently without redundancy.'},
  'Completeness': {'score': 5,
   'feedback': 'All key skill categories are present, well-categorized, and provide extensive detail, fulfilling expectations.'}}}